# Fusion: Text + WavLM + Whisper (LOBO-first, anti-overfit rewrite)

Rewritten 2026-04-21 after the first whisper run exposed multiple audios5-tuned configs as potentially-overfit. Every weight, threshold, and fusion choice is now made on **LOBO** (leave-one-training-batch-out) predictions. `TEST_FOLDER` (audios5) is evaluated **exactly once** at the end using those frozen configs.

## Base models (registry — toggle via config)
| name | features | head |
|---|---|---|
| `text_rf` | ~40 stylometric + formal_ai + disfluency + pause feats | RandomForest |
| `text_top5` | mattr, mtld, avg_word_length, ttr, formal_transition_count | XGBoost |
| `wavlm_wp` | frozen WavLM-base-plus whole-audio mean-pool (768d) | XGBoost |
| `wavlm_sp` | WavLM segmented mean+std pool (1536d) | XGBoost |
| `whisper_wp` | frozen Whisper-medium whole-audio mean-pool (1024d) | XGBoost |

## Anti-overfit rules
1. No threshold or α is picked on `TEST_FOLDER`.
2. **LOBO**: refit each base model leaving one training batch out, score the held batch, concatenate → one OOF proba vector per model covering all train rows.
3. All fusion search (2-way pair sweeps, 3-way simplex grid, stacking) happens on LOBO probas.
4. Frozen configs applied to test exactly once.
5. Any fusion where `|lobo_f1 - test_f1| > 0.03` is **flagged as overfit** and distrusted.
6. Second independent check: cross-batch swap eval (audios2↔4 and 2→5, 4→5).

## Outputs
- `checkpoints_fusion/lobo_metrics.csv` — realistic per-fusion numbers on unseen training data
- `checkpoints_fusion/test_oneshot_metrics.csv` — audios5 numbers at frozen config (+ gap vs LOBO)
- `checkpoints_fusion/cross_batch.csv` — swap-eval table with whisper included
- `checkpoints_fusion/frozen_configs.json` — weights + thresholds for every candidate
- `checkpoints_fusion/audios5_full_predictions.csv` + `review_audios5/` — per-file + misclassified copy


In [ ]:
# ================================================================
# CONFIGURATION
# ================================================================
from pathlib import Path

TRAIN_FOLDERS = ["audios2", "audios4"]
TEST_FOLDER   = "audios5"

# Base model registry switches (extend/disable without touching downstream code)
USE_TEXT_TOP5 = 1   # XGB on 5-feature stylometric core (surprising audios5 winner; LOBO will re-test)
USE_WAVLM_SP  = 1   # segmented WavLM (mean+std); historically weak on LOBO — kept as control
USE_WHISPER   = 1   # needs {folder}_whisper_whole.csv from encoder_comparison.ipynb

# Text features used for the broad text_rf head
TEXT_GROUPS    = ['stylometric', 'formal_ai', 'disfluency', 'pause']
TOP5_TEXT_FEATS = ['mattr', 'mtld', 'avg_word_length', 'ttr', 'formal_transition_count']

PREC_TARGETS = [0.80, 0.85, 0.90, 0.95]
RANDOM_SEED  = 42
CV_FOLDS     = 5   # used only if there is a single training batch (LOBO fallback)

NB_DIR   = Path('.').resolve()
SAVE_DIR = NB_DIR / 'checkpoints_fusion'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

LABEL_MAP = {
    'read':1,'cheating':1,'reading':1,'scripted':1,'yes':1,'1':1,1:1,
    'spontaneous':0,'not cheating':0,'not_cheating':0,'no':0,'0':0,0:0,'genuine':0,
}

_active = ['text_rf'] + (['text_top5'] if USE_TEXT_TOP5 else []) + ['wavlm_wp'] + \
          (['wavlm_sp'] if USE_WAVLM_SP else []) + (['whisper_wp'] if USE_WHISPER else [])
print(f'Train batches: {TRAIN_FOLDERS}')
print(f'Test batch   : {TEST_FOLDER}')
print(f'Active base models: {_active}')

In [ ]:
import json, itertools, warnings
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              confusion_matrix)
warnings.filterwarnings('ignore')
np.random.seed(RANDOM_SEED)

GROUPS = {
    'disfluency':  ['filler_rate','filler_count','repetition_rate','repair_rate',
                    'discourse_marker_rate','hedge_rate'],
    'stylometric': ['ttr','mattr','mtld','complex_word_rate','avg_word_length',
                    'n_words','n_unique_words','avg_sentence_length','std_sentence_length',
                    'fragment_rate','n_sentences','self_ref_rate',
                    'noun_rate','verb_rate','adj_rate'],
    'pause':       ['pause_mean','pause_std','pause_median','pause_skew','long_pause_rate',
                    'pause_ratio','n_pauses','pause_regularity',
                    'pause_before_content_ratio','pause_before_function_ratio',
                    'mid_phrase_pause_rate','words_per_sec','articulation_rate',
                    'initial_pause','longest_pause'],
    'formal_ai':   ['formal_transition_count','formal_transition_rate',
                    'ai_phrase_count','ai_phrase_rate'],
}
TEXT_FEATURES = [f for g in TEXT_GROUPS for f in GROUPS[g]]
print(f'Candidate text features: {len(TEXT_FEATURES)}')

## 1. Load cached features (per batch)

One dict `batches[name]` → dataframe with filename, label_int, all features, batch tag. Feature column lists are taken from the first training batch (all batches must share the same schema).

In [ ]:
def load_gt(name):
    gt = pd.read_csv(NB_DIR / f'{name}GT.csv')
    fn_col  = next(c for c in gt.columns if c.lower() in ('filename','file','name'))
    lbl_col = next(c for c in gt.columns if c.lower() in ('label','class','cheating','gt','label_int','ground_truth'))
    gt = gt.rename(columns={fn_col:'filename', lbl_col:'label_raw'})
    gt['label_int'] = gt['label_raw'].map(
        lambda x: LABEL_MAP.get(x, LABEL_MAP.get(str(x).lower().strip(), -1)))
    return gt[gt['label_int'].isin([0,1])][['filename','label_int']]

def load_folder(name):
    gt = load_gt(name)
    text_csv = NB_DIR / f'{name}_features.csv'
    wp_csv   = NB_DIR / f'{name}_whole_pretrained.csv'
    for p in (text_csv, wp_csv):
        if not p.exists(): raise FileNotFoundError(p)
    df = (gt.merge(pd.read_csv(text_csv), on='filename', how='inner')
            .merge(pd.read_csv(wp_csv),   on='filename', how='inner', suffixes=('','_wp')))
    if USE_WAVLM_SP:
        sp_csv = NB_DIR / f'{name}_seg_pretrained.csv'
        if not sp_csv.exists(): raise FileNotFoundError(sp_csv)
        df = df.merge(pd.read_csv(sp_csv), on='filename', how='inner', suffixes=('','_sp'))
    if USE_WHISPER:
        wh_csv = NB_DIR / f'{name}_whisper_whole.csv'
        if not wh_csv.exists(): raise FileNotFoundError(f'{wh_csv} (need whisper cache)')
        df = df.merge(pd.read_csv(wh_csv), on='filename', how='inner', suffixes=('','_wh'))
    df['batch'] = name
    return df

batches = {b: load_folder(b) for b in TRAIN_FOLDERS + [TEST_FOLDER]}

_first = batches[TRAIN_FOLDERS[0]]
wp_cols = [c for c in _first.columns if c.startswith('wavlm_')
           and not c.startswith('wavlm_mean_') and not c.startswith('wavlm_std_')]
sp_cols = [c for c in _first.columns if c.startswith('wavlm_mean_') or c.startswith('wavlm_std_')] if USE_WAVLM_SP else []
wh_cols = [c for c in _first.columns if c.startswith('whisper_')] if USE_WHISPER else []
text_cols = [c for c in TEXT_FEATURES if c in _first.columns]

def X_text(df):  return df[text_cols].fillna(0).values
def X_top5(df):  return df[TOP5_TEXT_FEATS].fillna(0).values
def X_wp(df):    return df[wp_cols].fillna(0).values
def X_sp(df):    return df[sp_cols].fillna(0).values
def X_wh(df):    return df[wh_cols].fillna(0).values

print(f'Text feats: {len(text_cols)}  | WP: {len(wp_cols)}' +
      (f'  | SP: {len(sp_cols)}' if USE_WAVLM_SP else '') +
      (f'  | Whisper: {len(wh_cols)}' if USE_WHISPER else ''))
for name, df in batches.items():
    y = df['label_int'].values
    print(f'  {name}: n={len(df):4d}  cheat={int((y==1).sum()):3d}  honest={int((y==0).sum()):3d}')

## 2. Evaluation helpers

`best_f1_on(p, y)` → best-F1 threshold (step 0.01). `metrics_at(p, y, thr)` → prec/rec/f1/confusion. `rec_at_prec(p, y)` → max recall achievable at each precision target.

In [ ]:
def best_f1_on(proba, y, thr_grid=np.arange(0.20, 0.81, 0.01)):
    best_f1, best_thr = -1.0, 0.5
    for thr in thr_grid:
        f = f1_score(y, (proba >= thr).astype(int), zero_division=0)
        if f > best_f1: best_f1, best_thr = f, thr
    return float(best_thr), float(best_f1)

def metrics_at(proba, y, thr):
    pred = (proba >= thr).astype(int)
    cm = confusion_matrix(y, pred, labels=[0,1])
    return dict(
        thr=round(float(thr), 3),
        precision=round(precision_score(y, pred, zero_division=0), 4),
        recall   =round(recall_score   (y, pred, zero_division=0), 4),
        f1       =round(f1_score       (y, pred, zero_division=0), 4),
        tp=int(cm[1,1]), fp=int(cm[0,1]), fn=int(cm[1,0]), tn=int(cm[0,0]),
    )

def rec_at_prec(proba, y, targets=PREC_TARGETS, min_tp=3):
    out = {}
    for t in targets:
        best_rec, best_thr = None, None
        for thr in np.arange(0.99, 0.10, -0.01):
            pred = (proba >= thr).astype(int)
            cm = confusion_matrix(y, pred, labels=[0,1])
            if cm[1,1] < min_tp: continue
            p = precision_score(y, pred, zero_division=0)
            r = recall_score   (y, pred, zero_division=0)
            if p >= t and (best_rec is None or r > best_rec):
                best_rec, best_thr = r, thr
        out[f'rec@P{int(t*100)}'] = round(best_rec, 4) if best_rec is not None else None
        out[f'thr@P{int(t*100)}'] = round(float(best_thr), 3) if best_thr is not None else None
    return out

def evaluate_proba(proba, y, name):
    thr, _ = best_f1_on(proba, y)
    return {'method': name, **metrics_at(proba, y, thr), **rec_at_prec(proba, y)}

## 3. Base-model registry + fit/score utility

`build_registry(spw)` returns `{name: (X_fn, make_clf)}` for the active models. Built per-fit so `scale_pos_weight` matches the actual positive count of that fit's training data (critical for LOBO, where each fold has a different class balance).

In [ ]:
def make_rf():
    return RandomForestClassifier(
        n_estimators=500, max_depth=8, min_samples_leaf=3,
        class_weight='balanced', n_jobs=-1, random_state=RANDOM_SEED)

def make_xgb(n_feats, spw):
    colsample = 0.3 if n_feats > 500 else 0.8
    return xgb.XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=colsample, min_child_weight=3,
        scale_pos_weight=spw, eval_metric='logloss', random_state=RANDOM_SEED)

def build_registry(spw):
    reg = {
        'text_rf':  (X_text, make_rf),
        'wavlm_wp': (X_wp,   lambda: make_xgb(len(wp_cols), spw)),
    }
    if USE_TEXT_TOP5:
        reg['text_top5'] = (X_top5, lambda: make_xgb(5, spw))
    if USE_WAVLM_SP:
        reg['wavlm_sp'] = (X_sp,    lambda: make_xgb(len(sp_cols), spw))
    if USE_WHISPER:
        reg['whisper_wp'] = (X_wh,  lambda: make_xgb(len(wh_cols), spw))
    return reg

def fit_and_score(X_fn, make_clf, df_tr, df_te):
    Xt, yt = X_fn(df_tr), df_tr['label_int'].values
    Xe     = X_fn(df_te)
    sc = StandardScaler().fit(Xt)
    m  = make_clf()
    m.fit(sc.transform(Xt), yt)
    return m.predict_proba(sc.transform(Xe))[:, 1]

print('Registry built per-fit with correct scale_pos_weight.')

## 4. LOBO predictions on training batches

For each training batch as holdout → refit each base model on the remaining training batches → score the held-out batch. Concatenate the holdout predictions across folds → one `lobo_scores[model]` vector per base model, aligned with `lobo_y` / `lobo_fn` / `lobo_batch`.

These are the numbers that matter for generalisation.

In [ ]:
def stratified_oof(df, X_fn, make_clf, folds=CV_FOLDS):
    """Fallback: 5-fold stratified OOF on a single batch (used only if TRAIN_FOLDERS has 1 entry)."""
    y = df['label_int'].values
    X = X_fn(df)
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=RANDOM_SEED)
    out = np.zeros(len(y))
    for tr, va in skf.split(X, y):
        sc = StandardScaler().fit(X[tr])
        m  = make_clf()
        m.fit(sc.transform(X[tr]), y[tr])
        out[va] = m.predict_proba(sc.transform(X[va]))[:, 1]
    return out

lobo_scores = {}
lobo_y_parts, lobo_fn_parts, lobo_batch_parts = [], [], []

for held in TRAIN_FOLDERS:
    df_held = batches[held]
    other_names = [b for b in TRAIN_FOLDERS if b != held]
    df_train = pd.concat([batches[b] for b in other_names], ignore_index=True) if other_names else None

    if df_train is not None:
        y_train = df_train['label_int'].values
        spw_l = (y_train==0).sum() / max((y_train==1).sum(), 1)
        reg = build_registry(spw_l)
        print(f'LOBO held={held}  n_train={len(df_train)}  n_held={len(df_held)}  spw={spw_l:.2f}')
        for name, (X_fn, make_clf) in reg.items():
            p = fit_and_score(X_fn, make_clf, df_train, df_held)
            lobo_scores.setdefault(name, []).append(p)
    else:
        y_held = df_held['label_int'].values
        spw_l = (y_held==0).sum() / max((y_held==1).sum(), 1)
        reg = build_registry(spw_l)
        print(f'Single train batch -> 5-fold OOF fallback on {held}')
        for name, (X_fn, make_clf) in reg.items():
            p = stratified_oof(df_held, X_fn, make_clf)
            lobo_scores.setdefault(name, []).append(p)

    lobo_y_parts.append(df_held['label_int'].values)
    lobo_fn_parts.append(df_held['filename'].values)
    lobo_batch_parts.append(np.full(len(df_held), held))

lobo_y     = np.concatenate(lobo_y_parts)
lobo_fn    = np.concatenate(lobo_fn_parts)
lobo_batch = np.concatenate(lobo_batch_parts)
for k in lobo_scores:
    lobo_scores[k] = np.concatenate(lobo_scores[k])

print(f'\nLOBO rows: {len(lobo_y)}  cheat={int((lobo_y==1).sum())}  honest={int((lobo_y==0).sum())}')
print(f'Models in LOBO: {list(lobo_scores)}')

## 5. LOBO base-model metrics

Realistic per-model numbers. If `whisper_wp` has much better LOBO F1 than `wavlm_wp`, expect whisper to dominate fusion; if they're close on LOBO but whisper wins big on audios5 alone, that win is suspicious.

In [ ]:
lobo_base_rows = [evaluate_proba(lobo_scores[m], lobo_y, f'lobo:base:{m}') for m in lobo_scores]
lobo_base_df = pd.DataFrame(lobo_base_rows)
print('LOBO base-model metrics:')
print(lobo_base_df[['method','thr','precision','recall','f1',
                    'rec@P80','rec@P85','rec@P90','rec@P95']].to_string(index=False))

## 6. Fusion weight search on LOBO

For every pair (2-way) and every triple (3-way, 0.1-step simplex grid), sweep weights on LOBO probas. Pick the (weights, threshold) that maximise **LOBO F1**. Freeze them. Also fit a stacking `meta_logreg` on LOBO probas (with nested 5-fold OOF to pick its own best-F1 threshold fairly).

Every entry in `frozen` becomes a candidate to apply to `audios5` in Section 7.

In [ ]:
frozen = []  # each entry: tag, members, weights, thr, lobo_f1, lobo_prec, lobo_rec, lobo_rec@P*, proba_lobo

def grid_2way(step=0.05):
    return [(round(w, 2), round(1-w, 2)) for w in np.arange(0.0, 1.001, step)]

def grid_3way(step=0.1):
    out = []
    for w1 in np.arange(0, 1.001, step):
        for w2 in np.arange(0, 1.001 - w1 + 1e-9, step):
            w3 = 1.0 - w1 - w2
            if w3 < -1e-9: continue
            out.append((round(w1, 2), round(w2, 2), round(max(0, w3), 2)))
    return out

def record(tag, members, weights, proba_lobo):
    thr, _ = best_f1_on(proba_lobo, lobo_y)
    m = metrics_at(proba_lobo, lobo_y, thr)
    rp = rec_at_prec(proba_lobo, lobo_y)
    frozen.append({
        'tag': tag, 'members': list(members), 'weights': list(weights),
        'thr': m['thr'], 'lobo_f1': m['f1'],
        'lobo_prec': m['precision'], 'lobo_rec': m['recall'],
        'lobo_rec@P85': rp.get('rec@P85'),
        'lobo_rec@P90': rp.get('rec@P90'),
        'lobo_rec@P95': rp.get('rec@P95'),
        'proba_lobo': proba_lobo,
    })

model_names = list(lobo_scores)

# ---- 2-way weighted averages ----
for a, b in itertools.combinations(model_names, 2):
    best = None
    for wa, wb in grid_2way(step=0.05):
        p = wa * lobo_scores[a] + wb * lobo_scores[b]
        thr, f1 = best_f1_on(p, lobo_y)
        if best is None or f1 > best['f1']:
            best = {'w': (wa, wb), 'p': p, 'f1': f1}
    record(f'wavg:{a}+{b}', [a, b], best['w'], best['p'])

# ---- 3-way weighted averages ----
if len(model_names) >= 3:
    for trio in itertools.combinations(model_names, 3):
        best = None
        for w in grid_3way(step=0.1):
            p = sum(wi * lobo_scores[m] for wi, m in zip(w, trio))
            thr, f1 = best_f1_on(p, lobo_y)
            if best is None or f1 > best['f1']:
                best = {'w': w, 'p': p, 'f1': f1}
        record(f'wavg:{"+".join(trio)}', list(trio), best['w'], best['p'])

# ---- Stacking: meta-logreg on LOBO probas, nested 5-fold OOF for its threshold ----
X_meta_lobo = np.column_stack([lobo_scores[m] for m in model_names])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
p_meta_oof = np.zeros(len(lobo_y))
for tr, va in skf.split(X_meta_lobo, lobo_y):
    mm = LogisticRegression(C=1.0, class_weight='balanced', max_iter=2000, random_state=RANDOM_SEED)
    mm.fit(X_meta_lobo[tr], lobo_y[tr])
    p_meta_oof[va] = mm.predict_proba(X_meta_lobo[va])[:, 1]

meta_lr = LogisticRegression(C=1.0, class_weight='balanced', max_iter=2000, random_state=RANDOM_SEED)
meta_lr.fit(X_meta_lobo, lobo_y)

thr_m, f1_m = best_f1_on(p_meta_oof, lobo_y)
mm_metrics = metrics_at(p_meta_oof, lobo_y, thr_m)
mm_rp      = rec_at_prec(p_meta_oof, lobo_y)
frozen.append({
    'tag': 'stack:meta_logreg',
    'members': model_names,
    'weights': [float(w) for w in np.round(meta_lr.coef_[0], 4)],
    'meta_intercept': float(meta_lr.intercept_[0]),
    'thr': mm_metrics['thr'], 'lobo_f1': mm_metrics['f1'],
    'lobo_prec': mm_metrics['precision'], 'lobo_rec': mm_metrics['recall'],
    'lobo_rec@P85': mm_rp.get('rec@P85'),
    'lobo_rec@P90': mm_rp.get('rec@P90'),
    'lobo_rec@P95': mm_rp.get('rec@P95'),
    'proba_lobo': p_meta_oof,
})

lobo_df = pd.DataFrame([{k: v for k, v in cfg.items() if k != 'proba_lobo'} for cfg in frozen])
lobo_df = lobo_df.sort_values('lobo_f1', ascending=False).reset_index(drop=True)
print(f'\nTotal fusion candidates: {len(lobo_df)}')
print('\nTop 20 by LOBO F1:')
cols = ['tag', 'weights', 'thr', 'lobo_f1', 'lobo_prec', 'lobo_rec',
        'lobo_rec@P85', 'lobo_rec@P90', 'lobo_rec@P95']
print(lobo_df[cols].head(20).to_string(index=False))

## 7. Error overlap on LOBO (Jaccard)

Low Jaccard = complementary models = fusion should work. High Jaccard = redundant → fusion won't add much.

In [ ]:
errors = {}
for name, p in lobo_scores.items():
    thr, _ = best_f1_on(p, lobo_y)
    pred = (p >= thr).astype(int)
    errors[name] = set(np.where(pred != lobo_y)[0])

names = list(errors.keys())
print('LOBO error set sizes:', {n: len(errors[n]) for n in names})
print('\nJaccard overlap of LOBO errors:')
print(f'{"":<12}' + ''.join(f'{n:>14}' for n in names))
for a in names:
    row = [f'{a:<12}']
    for b in names:
        if a == b:
            row.append(f'{1.0:>14.3f}')
        else:
            inter = len(errors[a] & errors[b])
            union = len(errors[a] | errors[b])
            row.append(f'{inter/max(union,1):>14.3f}')
    print(''.join(row))

## 8. Apply frozen configs to TEST (audios5) — one-shot

Fit each base model on the FULL training set, score `audios5`, then apply each frozen (weights, threshold) exactly once. `gap_f1 = lobo_f1 - test_f1`.

**Rule:** if `|gap_f1| > 0.03` the config is flagged — either overfit to train (positive gap) or the test batch is materially out-of-distribution (negative gap). Either way, don't blindly deploy.

In [ ]:
df_tr_all = pd.concat([batches[b] for b in TRAIN_FOLDERS], ignore_index=True)
df_te     = batches[TEST_FOLDER]
y_te      = df_te['label_int'].values
fn_te     = df_te['filename'].values
spw_full  = (df_tr_all['label_int']==0).sum() / max((df_tr_all['label_int']==1).sum(), 1)
full_reg  = build_registry(spw_full)

test_scores = {}
for name, (X_fn, make_clf) in full_reg.items():
    test_scores[name] = fit_and_score(X_fn, make_clf, df_tr_all, df_te)
print(f'Test scored for: {list(test_scores)}')
print(f'Test rows: {len(y_te)}  cheat={int((y_te==1).sum())}  honest={int((y_te==0).sum())}')

# Also keep the per-base test proba for reference evaluation
test_base_rows = [evaluate_proba(test_scores[m], y_te, f'test:base:{m}') for m in test_scores]
test_base_df   = pd.DataFrame(test_base_rows)
print('\nTest base-model metrics (reference, NOT used for threshold selection):')
print(test_base_df[['method','thr','precision','recall','f1',
                    'rec@P80','rec@P85','rec@P90','rec@P95']].to_string(index=False))

# Apply frozen configs to test
rows = []
for cfg in frozen:
    members = cfg['members']; weights = cfg['weights']; thr = cfg['thr']
    if cfg['tag'] == 'stack:meta_logreg':
        X_meta_te = np.column_stack([test_scores[m] for m in members])
        p_te = meta_lr.predict_proba(X_meta_te)[:, 1]
    else:
        p_te = sum(w * test_scores[m] for w, m in zip(weights, members))
    m_te = metrics_at(p_te, y_te, thr)
    rp   = rec_at_prec(p_te, y_te)
    rows.append({
        'tag': cfg['tag'], 'weights': weights, 'thr': thr,
        'lobo_f1': cfg['lobo_f1'],
        'test_f1': m_te['f1'], 'test_prec': m_te['precision'], 'test_rec': m_te['recall'],
        'gap_f1': round(cfg['lobo_f1'] - m_te['f1'], 4),
        'test_rec@P85': rp.get('rec@P85'),
        'test_rec@P90': rp.get('rec@P90'),
        'test_rec@P95': rp.get('rec@P95'),
        'proba_test':  p_te,
    })

one_shot_df = pd.DataFrame(rows).sort_values('lobo_f1', ascending=False).reset_index(drop=True)
show_cols = ['tag','weights','thr','lobo_f1','test_f1','gap_f1',
             'test_prec','test_rec','test_rec@P85','test_rec@P90','test_rec@P95']
print('\nFrozen configs → test (sorted by LOBO F1):')
print(one_shot_df[show_cols].head(20).to_string(index=False))

flagged = one_shot_df[one_shot_df['gap_f1'].abs() > 0.03]
print(f'\nFlagged (|gap_f1| > 0.03): {len(flagged)} / {len(one_shot_df)}')
if len(flagged):
    print(flagged[show_cols].to_string(index=False))

## 9. Cross-batch swap evaluation (secondary sanity)

Independent of LOBO. Train on ONE batch, test on another. If a fusion wins on `audios2↔audios4` AND `2→5` AND `4→5`, it's robust. If it only wins on `→audios5`, reject.

Now includes every base model in `build_registry()` (whisper + wavlm_sp if enabled) and a best-F1 α sweep per pair.

In [ ]:
def swap_eval(train_name, test_name):
    tr = batches[train_name]; te = batches[test_name]
    yt, ye = tr['label_int'].values, te['label_int'].values
    spw_l = (yt==0).sum() / max((yt==1).sum(), 1)
    reg = build_registry(spw_l)

    probs = {}
    for name, (X_fn, make_clf) in reg.items():
        probs[name] = fit_and_score(X_fn, make_clf, tr, te)

    rows = []
    for n, p in probs.items():
        thr, _ = best_f1_on(p, ye)
        rows.append({'train': train_name, 'test': test_name,
                     'method': f'base:{n}', **metrics_at(p, ye, thr)})

    # best-F1 α for every pair on this split (sanity view, NOT deployment)
    for a, b in itertools.combinations(probs.keys(), 2):
        best = None
        for wa, wb in grid_2way(step=0.1):
            p = wa * probs[a] + wb * probs[b]
            thr, f1 = best_f1_on(p, ye)
            if best is None or f1 > best['f1']:
                best = {'w': (wa, wb), 'thr': thr, 'f1': f1, 'p': p}
        rows.append({'train': train_name, 'test': test_name,
                     'method': f'wavg:{a}+{b}@a={best["w"][0]}',
                     **metrics_at(best['p'], ye, best['thr'])})
    return rows

swap_rows = []
for a, b in [('audios2','audios4'), ('audios4','audios2'),
             ('audios2','audios5'), ('audios4','audios5')]:
    if a in batches and b in batches:
        swap_rows.extend(swap_eval(a, b))

swap_df = pd.DataFrame(swap_rows)
print('Cross-batch generalisation (best-F1 on each split):')
print(swap_df[['train','test','method','precision','recall','f1']].to_string(index=False))

## 10. Isotonic calibration on the top LOBO fusion (optional)

Fit isotonic on LOBO (score → true positive rate) and apply to test. After calibration, `score ≥ 0.85` should mean ≈85% precision — useful for picking deployment thresholds by precision target.

In [ ]:
top_tag = lobo_df.iloc[0]['tag']
top_cfg = next(cfg for cfg in frozen if cfg['tag'] == top_tag)
print(f'Calibrating top LOBO fusion: {top_tag}')

if top_tag == 'stack:meta_logreg':
    p_lo = top_cfg['proba_lobo']
    p_te = meta_lr.predict_proba(np.column_stack([test_scores[m] for m in top_cfg['members']]))[:, 1]
else:
    p_lo = sum(w * lobo_scores[m] for w, m in zip(top_cfg['weights'], top_cfg['members']))
    p_te = sum(w * test_scores[m] for w, m in zip(top_cfg['weights'], top_cfg['members']))

iso = IsotonicRegression(out_of_bounds='clip').fit(p_lo, lobo_y)
p_lo_c = iso.transform(p_lo)
p_te_c = iso.transform(p_te)

print('\nReliability check — apply calibrated-score threshold = target precision, measure ACTUAL precision:')
print(f'{"thr":>6}  {"test_prec":>10}  {"test_rec":>9}  {"tp":>4}  {"fp":>4}  {"fn":>4}')
for t in [0.50, 0.60, 0.70, 0.80, 0.85, 0.90, 0.95]:
    pred = (p_te_c >= t).astype(int)
    cm = confusion_matrix(y_te, pred, labels=[0,1])
    if cm[1,1] < 3:
        continue
    prec = precision_score(y_te, pred, zero_division=0)
    rec  = recall_score(y_te, pred, zero_division=0)
    print(f'{t:>6.2f}  {prec:>10.4f}  {rec:>9.4f}  {cm[1,1]:>4}  {cm[0,1]:>4}  {cm[1,0]:>4}')

## 11. Save frozen configs + misclassification review

Persists LOBO table, one-shot test table, cross-batch table, and the per-file predictions for the top LOBO fusion. Copies misclassified audio files under `review_audios5/{FP,FN,CORRECT_BORDERLINE}/` for manual review.

In [ ]:
import shutil

# Save tables
lobo_df.to_csv(SAVE_DIR / 'lobo_metrics.csv', index=False)
one_shot_df.drop(columns=['proba_test']).to_csv(SAVE_DIR / 'test_oneshot_metrics.csv', index=False)
swap_df.to_csv(SAVE_DIR / 'cross_batch.csv', index=False)

# Save frozen configs (drop proba arrays for clean json)
serial_frozen = []
for cfg in frozen:
    d = {k: v for k, v in cfg.items() if k != 'proba_lobo'}
    serial_frozen.append(d)
with open(SAVE_DIR / 'frozen_configs.json', 'w') as f:
    json.dump(serial_frozen, f, indent=2, default=str)

# Per-file review for the top LOBO fusion
top_row = one_shot_df.iloc[0]
proba_test = top_row['proba_test']; thr = top_row['thr']

review = pd.DataFrame({
    'filename': fn_te,
    'current_gt': y_te.astype(int),
    'score':     np.round(proba_test, 4),
    'score_cal': np.round(p_te_c, 4),
    'pred':      (proba_test >= thr).astype(int),
})
for m, p in test_scores.items():
    review[f'p_{m}'] = np.round(p, 4)

def err_type(row):
    if row['current_gt'] == row['pred']:
        if abs(row['score'] - thr) <= 0.08: return 'CORRECT_BORDERLINE'
        return 'CORRECT'
    return 'FP' if row['current_gt'] == 0 else 'FN'

review['error_type'] = review.apply(err_type, axis=1)
review = review.sort_values('score', ascending=False).reset_index(drop=True)
review['rank'] = review.index + 1
review['correct_gt'] = ''
review['notes'] = ''

col_order = ['rank', 'filename', 'current_gt', 'score', 'score_cal', 'pred', 'error_type'] \
            + [f'p_{m}' for m in test_scores] \
            + ['correct_gt', 'notes']
review = review[col_order]

full_pred_path = SAVE_DIR / 'audios5_full_predictions.csv'
review.to_csv(full_pred_path, index=False)
print(f'Full predictions -> {full_pred_path}')

# Misclassified copy
REVIEW_DIR = NB_DIR / 'review_audios5'
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
misc = review[review['error_type'].isin(['FP','FN','CORRECT_BORDERLINE'])].copy()
misc = misc.sort_values(['error_type','score'], ascending=[True, False]).reset_index(drop=True)
misc.to_csv(REVIEW_DIR / 'audios5_misclassified.csv', index=False)

src_candidates = [NB_DIR / TEST_FOLDER, NB_DIR.parent / TEST_FOLDER]
src_dir = next((p for p in src_candidates if p.exists()), None)
copied, missing = 0, []
if src_dir:
    for _, row in misc.iterrows():
        fn = row['filename']
        sub = REVIEW_DIR / row['error_type']; sub.mkdir(parents=True, exist_ok=True)
        src_file = src_dir / fn
        if not src_file.exists():
            for ext in ('.wav','.mp3','.m4a','.flac','.ogg'):
                alt = src_dir / (fn + ext)
                if alt.exists(): src_file = alt; break
        if src_file.exists():
            dest = sub / f"r{int(row['rank']):03d}_s{row['score']:.3f}_gt{int(row['current_gt'])}_{src_file.name}"
            shutil.copy2(src_file, dest); copied += 1
        else:
            missing.append(fn)
    print(f'Copied {copied} misclassified files to {REVIEW_DIR} (missing: {len(missing)})')
else:
    print(f'[WARN] {TEST_FOLDER}/ not found — copy files manually.')

print('\n' + '='*70)
print('Summary — top LOBO fusion')
print('='*70)
print(f'  tag     : {top_row["tag"]}')
print(f'  weights : {top_row["weights"]}')
print(f'  thr     : {top_row["thr"]}')
print(f'  LOBO F1 : {top_row["lobo_f1"]:.4f}')
print(f'  Test F1 : {top_row["test_f1"]:.4f}   (gap: {top_row["gap_f1"]:+.4f})')
print(f'  Test P/R: {top_row["test_prec"]:.4f} / {top_row["test_rec"]:.4f}')
print(f'  rec@P85={top_row["test_rec@P85"]}  rec@P90={top_row["test_rec@P90"]}  rec@P95={top_row["test_rec@P95"]}')
print(f'\nArtifacts in: {SAVE_DIR}')